# Benchmarking Classical Machine Learning Model on AptaBench
This notebook benchmarks **traditional ML model** (LightGBM) on the AptaBench dataset, focusing on feature engineering, cross-validation, and baseline predictive performance.

## 1. Imports & Config

In [1]:
import torch
torch.__version__

'2.6.0+cu124'

In [1]:
# Add project root to sys.path so "src" is importable
import sys
from pathlib import Path

root = Path.cwd()
while not (root / "src").exists() and root.parent != root:
    root = root.parent

sys.path.insert(0, str(root))

In [2]:
# Fix NumPy version compatibility with RDKit
# RDKit requires NumPy < 2.0, but transformers may install NumPy 2.x
import numpy as np
if np.__version__.startswith('2.'):
    import subprocess
    import sys
    print(f"NumPy version {np.__version__} detected. Downgrading to < 2.0 for RDKit compatibility...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "numpy<2.0", "--force-reinstall", "-q"])
    print("NumPy downgraded. Please restart the kernel and run again.")
    raise RuntimeError("NumPy downgraded. Please restart the kernel (Kernel -> Restart Kernel) and run cells again.")
else:
    print(f"NumPy version {np.__version__} is compatible with RDKit.")


NumPy version 1.26.4 is compatible with RDKit.


In [3]:
# === Imports ===
import os
import pandas as pd
from src.models.screening import screen_lgbm_optuna, load_splits
# --- Aptamer encoders ---
from src.encoders.aptamer_encoders import (
    onehot_with_type_bit,
    kmer_freq_with_type_bit,
    gena_embed,
    dnabert2_embed
)

# --- Molecule encoders ---
from src.encoders.molecule_encoders import (
    morgan_fp,
    maccs_fp,
    physchem_descriptors,
    chemberta_embed,
    unimol_embed,
    molformer_embed,
    clamp_embed
)

# Paths
DATASET_PATH = "../dataset/AptaBench_dataset_v2.csv" 
SPLITS_DIR = "../dataset/splits"

c:\Users\m19er\AptaBench\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Failed to find the pandas get_adjustment() function to patch
Failed to patch pandas - PandasTools will have limited functionality


In [4]:
# Load dataset
df = pd.read_csv(DATASET_PATH)
print("Dataset loaded:", df.shape)
display(df.head())

# Check available splits
print("Available splits:", os.listdir(SPLITS_DIR))

Dataset loaded: (6413, 8)


,type,sequence,canonical_smiles,pKd_value,label,buffer,origin,source
0,DNA,GGGAGAATTCCCGCGGCAGAAGCCCACCTGGCTTTGAACTCTATGT...,Nc1c(S(=O)(=O)O)cc(Nc2ccc(Nc3nc(Cl)nc(Nc4ccccc...,4.000000,0,NaN,NaN,RSAPred
1,RNA,GGGAGAAUUCCCGCGGCGUUGGCCCAGGAUAAUAGGACGAAAUCCG...,Nc1c(S(=O)(=O)O)cc(Nc2ccc(S(=O)(=O)O)c(Nc3nc(C...,3.221849,0,NaN,NaN,RSAPred
2,RNA,GGGAAGGGAAGAAACUGCGGCUUCGGCCGGCUUCCC,Nc1ncnc2c1ncn2C1OC(COP(=O)(O)OP(=O)(O)OP(=O)(O...,5.397940,1,NaN,NaN,RSAPred
3,RNA,GGGAAGGGAAGAAACUGCGGCUUCGGCCGGCUUCCC,Nc1ncnc2c1ncn2C1OC(COP(=O)(O)O)C(O)C1O,8.301026,1,NaN,NaN,RSAPred
4,RNA,GGCGUGUAGGAUAUGCUUCGGCAGAAGGACACGCC,Cc1cc2nc3c(=O)[nH]c(=O)nc-3n(CC(O)C(O)C(O)COP(...,4.638272,1,NaN,NaN,RSAPred


Available splits: ['disjoint_aptamer.json', 'disjoint_molecule.json', 'stratified.json']


## 2. Encoder configurations

In [5]:
apt_cfgs = [
    {"name": "OneHot", "func": onehot_with_type_bit},
    {"name": "Kmer3", "func": kmer_freq_with_type_bit, "kwargs": {"k": 3}},
    {"name": "Kmer4", "func": kmer_freq_with_type_bit, "kwargs": {"k": 4}},
    {"name": "GENA", "func": gena_embed},
    {"name": "DNABERT2", "func": dnabert2_embed},
]

mol_cfgs = [
    {"name": "MorganFP", "func": morgan_fp},
    {"name": "MACCS", "func": maccs_fp},
    {"name": "PhysChem", "func": physchem_descriptors},
    {"name": "ChemBERTa", "func": chemberta_embed},
    {"name": "UniMol", "func": unimol_embed},
    {"name": "MolFormer", "func": molformer_embed},
    {"name": "CLAMP", "func": clamp_embed},
]

In [6]:
  from src.models.screening import precompute_and_save_embeddings
  precompute_and_save_embeddings(df, apt_cfgs, mol_cfgs)

Dataset hash: 556cdb84df79cb8068f04358678fd4df
Cache directory: notebooks\data\embeddings
Loading cached aptamer embeddings...
  ✓ Loaded OneHot: shape (6413, 865)
  ✓ Loaded Kmer3: shape (6413, 65)
  ✓ Loaded Kmer4: shape (6413, 257)
  ✓ Loaded GENA: shape (6413, 769)
  ✓ Loaded DNABERT2: shape (6413, 769)
Loading cached molecule embeddings...
  ✓ Loaded MorganFP: shape (6413, 2048)
  ✓ Loaded MACCS: shape (6413, 167)
  ✓ Loaded PhysChem: shape (6413, 196)
  ✓ Loaded ChemBERTa: shape (6413, 768)
  ✓ Loaded UniMol: shape (6413, 512)
  ✓ Loaded MolFormer: shape (6413, 768)
  ✓ Loaded CLAMP: shape (6413, 768)


({'OneHot': array([[0., 0., 1., ..., 0., 0., 0.],
         [0., 0., 1., ..., 0., 0., 1.],
         [0., 0., 1., ..., 0., 0., 1.],
         ...,
         [0., 0., 1., ..., 0., 0., 0.],
         [0., 0., 1., ..., 0., 0., 0.],
         [0., 0., 1., ..., 0., 0., 0.]]),
  'Kmer3': array([[0.02307692, 0.03076923, 0.02307692, ..., 0.01538462, 0.00769231,
          0.        ],
         [0.03100775, 0.03100775, 0.00775194, ..., 0.00775194, 0.        ,
          1.        ],
         [0.02941176, 0.02941176, 0.05882353, ..., 0.        , 0.        ,
          1.        ],
         ...,
         [0.        , 0.        , 0.02272727, ..., 0.        , 0.        ,
          0.        ],
         [0.        , 0.        , 0.02272727, ..., 0.04545455, 0.        ,
          0.        ],
         [0.        , 0.        , 0.        , ..., 0.04545455, 0.04545455,
          0.        ]]),
  'Kmer4': array([[0.00775194, 0.01550388, 0.        , ..., 0.00775194, 0.        ,
          0.        ],
         [0.01

In [12]:
!pip install optuna-integration[lightgbm]

In [ ]:
import gc

# Очищаем память перед началом
gc.collect()

# Запускаем по одному split_mode за раз
all_results = []

for split_mode in ["stratified", "disjoint_molecule", "disjoint_aptamer"]:
    print(f"\n{'='*60}")
    print(f"Processing split_mode: {split_mode}")
    print(f"{'='*60}\n")
    
    try:
        results_df = screen_lgbm_optuna(
            df,
            apt_cfgs,
            mol_cfgs,
            split_modes=(split_mode,),  # По одному за раз
            n_trials=10,  # УМЕНЬШЕНО с 10 для ускорения
            metric="roc_auc",
            splits_dir="../dataset/splits",
            use_cached_embeddings=True,  # Используем кэш эмбеддингов
        )
        all_results.append(results_df)
        
        # Сохраняем промежуточные результаты
        results_df.to_csv(f'lgbm_results_{split_mode}.csv', index=False)
        print(f"✓ Saved intermediate results for {split_mode}")
        
    except Exception as e:
        print(f"✗ Error processing {split_mode}: {e}")
        import traceback
        traceback.print_exc()
    
    # Очищаем память после каждого split_mode
    gc.collect()
    print(f"Memory cleared after {split_mode}\n")

# Объединяем все результаты
if all_results:
    results_df = pd.concat(all_results, ignore_index=True)
    results_df.to_csv('lgbm_results_screening_complete.csv', index=False)
    print("✓ All results saved to lgbm_results_screening_complete.csv")
else:
    print("✗ No results collected")



Processing split_mode: stratified



Apt encoders (stratified):   0%|          | 0/5 [00:00<?, ?it/s][I 2025-11-30 01:20:51,893] Trial 1 finished with value: 0.9414545189688874 and parameters: {'n_estimators': 436, 'learning_rate': 0.02895839011109895, 'num_leaves': 26, 'max_depth': 7, 'subsample': 0.7540118742058126, 'colsample_bytree': 0.646588622871202, 'reg_lambda': 1.2629895191299234}. Best is trial 1 with value: 0.9414545189688874.
[I 2025-11-30 01:21:32,143] Trial 2 finished with value: 0.9490634281231822 and parameters: {'n_estimators': 495, 'learning_rate': 0.038526323869859574, 'num_leaves': 39, 'max_depth': 14, 'subsample': 0.7906719278084016, 'colsample_bytree': 0.7495781544114503, 'reg_lambda': 0.16182687900218054}. Best is trial 2 with value: 0.9490634281231822.
[I 2025-11-30 01:21:48,854] Trial 3 finished with value: 0.883606258801635 and parameters: {'n_estimators': 236, 'learning_rate': 0.08917161961513166, 'num_leaves': 121, 'max_depth': 1, 'subsample': 0.6302294051790792, 'colsample_bytree': 0.685848482

## 3. 

In [3]:
results_df.to_csv('lgbm_results_screening_2.csv')

NameError: name 'results_df' is not defined

### Ускорение + Чекпоинты + Онлайн сохранение

**Управление памятью:**
1. **Ленивая загрузка эмбеддингов** - загружаем только нужные для текущей комбинации
2. **float32 вместо float64** - экономия ~50% памяти
3. **n_jobs=2** для LightGBM - баланс скорости и памяти
4. **Агрессивная очистка памяти** - после каждого trial, fold и комбинации

**Ускорение (в 2-3 раза быстрее):**
1. **Pruning (MedianPruner)** - раннее прекращение плохих trials (~30-50% ускорение)
2. **Early stopping в LightGBM** - остановка при отсутствии улучшения
3. **TPESampler** - умный поиск вместо случайного
4. **Optuna SQLite storage** - кэширование trials для перезапуска
5. **Все 5 folds** - используем все фолды для оценки (качество не теряется)

**Чекпоинты и перезапуск:**
1. **Автоматические чекпоинты** - после каждой комбинации энкодеров
2. **Продолжение с места остановки** - `resume=True` пропускает выполненные комбинации
3. **Онлайн сохранение** - результаты сохраняются после каждой итерации
4. **Optuna DB** - каждый study сохраняется в SQLite для перезапуска

**Результат:** В 2-3 раза быстрее + защита от падений + возможность перезапуска


In [13]:
results_df[results_df['split'] == 'stratified'].pivot(
    index="aptamer_encoder", columns="molecule_encoder", values="MCC mean"
)

molecule_encoder,ChemBERTa,MACCS,MolFormer,MorganFP,PhysChem,UniMol
aptamer_encoder,,,,,,
DNABERT2,0.768531,0.724158,0.766834,0.747607,0.755794,0.759408
GENA,0.768362,0.726029,0.765555,0.748566,0.758657,0.761412
Kmer3,0.774855,0.759361,0.778054,0.769518,0.775013,0.770112
Kmer4,0.776240,0.762679,0.779085,0.774509,0.779623,0.775212
OneHot,0.763922,0.761879,0.773023,0.772831,0.775607,0.762692


In [15]:
results_df[results_df['split'] == 'disjoint_aptamer'].pivot(
    index="aptamer_encoder", columns="molecule_encoder", values="MCC mean"
)

molecule_encoder,ChemBERTa,MACCS,MolFormer,MorganFP,PhysChem,UniMol
aptamer_encoder,,,,,,
DNABERT2,0.710313,0.675365,0.704688,0.683720,0.697151,0.708630
GENA,0.700830,0.668374,0.701985,0.660668,0.696946,0.699712
Kmer3,0.715607,0.697849,0.713297,0.700204,0.709971,0.717444
Kmer4,0.724430,0.710764,0.729047,0.716341,0.723574,0.726027
OneHot,0.714824,0.716003,0.720402,0.721243,0.719885,0.724340


In [ ]:
# УСКОРЕННАЯ ВЕРСИЯ С ЧЕКПОИНТАМИ И ОНЛАЙН СОХРАНЕНИЕМ
# - Чекпоинты после каждой комбинации энкодеров
# - Автоматическое продолжение с места остановки
# - Онлайн сохранение результатов после каждой итерации
# - Pruning + Early stopping для ускорения в 2-3 раза
# - Используем все 5 folds для оценки (качество не теряется)

import gc
from pathlib import Path

# Очищаем память перед началом
gc.collect()

# Запускаем по одному split_mode за раз
all_results = []

for split_mode in ["stratified", "disjoint_molecule", "disjoint_aptamer"]:
    print(f"\n{'='*60}")
    print(f"Processing split_mode: {split_mode}")
    print(f"{'='*60}\n")
    
    try:
        results_df = screen_lgbm_optuna(
            df,
            apt_cfgs,
            mol_cfgs,
            split_modes=(split_mode,),  # По одному за раз
            n_trials=10,  # Количество trials
            metric="roc_auc",
            splits_dir="../dataset/splits",
            use_cached_embeddings=True,  # Используем кэш эмбеддингов
            checkpoint_dir="notebooks/checkpoints",  # Директория для чекпоинтов
            results_file=f"notebooks/checkpoints/results_{split_mode}.csv",  # Онлайн сохранение
            resume=True,  # Продолжить с чекпоинта если есть
            use_pruning=True,  # Pruning для ускорения (~30-50%)
        )
        all_results.append(results_df)
        
        # Сохраняем финальные результаты
        results_df.to_csv(f'lgbm_results_{split_mode}.csv', index=False)
        print(f"✓ Saved final results for {split_mode}")
        
    except Exception as e:
        print(f"✗ Error processing {split_mode}: {e}")
        import traceback
        traceback.print_exc()
        # Пытаемся загрузить частичные результаты
        checkpoint_file = f"notebooks/checkpoints/results_{split_mode}.csv"
        if Path(checkpoint_file).exists():
            print(f"⚠ Загружаем частичные результаты из {checkpoint_file}")
            try:
                partial_df = pd.read_csv(checkpoint_file)
                all_results.append(partial_df)
            except:
                pass
    
    # Очищаем память после каждого split_mode
    gc.collect()
    print(f"Memory cleared after {split_mode}\n")

# Объединяем все результаты
if all_results:
    results_df = pd.concat(all_results, ignore_index=True)
    results_df.to_csv('lgbm_results_screening_complete.csv', index=False)
    print("✓ All results saved to lgbm_results_screening_complete.csv")
else:
    print("✗ No results collected")


In [16]:
results_df[results_df['split'] == 'disjoint_molecule'].pivot(
    index="aptamer_encoder", columns="molecule_encoder", values="MCC mean"
)

molecule_encoder,ChemBERTa,MACCS,MolFormer,MorganFP,PhysChem,UniMol
aptamer_encoder,,,,,,
DNABERT2,0.586702,0.639210,0.614147,0.642462,0.635453,0.629267
GENA,0.582859,0.629948,0.603005,0.636817,0.627775,0.608431
Kmer3,0.564354,0.626377,0.580366,0.627274,0.620082,0.599143
Kmer4,0.601937,0.637889,0.614110,0.648660,0.634019,0.631660
OneHot,0.585966,0.638199,0.643270,0.660474,0.619844,0.620263


In [15]:
import seaborn as sns